# Housing Affordability and Tenure Analysis

This notebook uses the prepared `master.csv` file to estimate survey-weighted housing affordability indicators by year, tenure group, income position, and urban geography.


## Data Sources

The household survey files used here are the cleaned Iranian Household Budget Survey files for 1394-1403 published by D-Learn: <https://d-learn.ir/iran-hbs/>. The page describes the files as cleaned and harmonized CSV/XLSX versions of the Statistical Center of Iran Household Expenditure and Income Survey.

CPI information is taken from the World Bank indicator `FP.CPI.TOTL.ZG` for Iran: <https://data.worldbank.org/indicator/FP.CPI.TOTL.ZG?end=2024&locations=IR&start=1960>. The local workbook `CPI.xlsx` is used as the project copy for the CPI merge.

The repository should treat the files in this folder as derived working data. If raw or cleaned survey files are not redistributed, the links above identify where the source data can be obtained.


## Scope and Limitations

The data are treated as repeated annual cross-sections, not as a household panel. Tenure groups are assigned from text patterns in the `Tenure` field, which is transparent and reproducible but still depends on the source naming convention. The adjusted housing-cost measure removes imputed rent for owner and mortgage households; this choice is appropriate for the burden measures used here, but alternative definitions may be useful for other research questions.

The CPI adjustment is used to compare values over time. Because the analysis combines local CPI data with household survey data, the exact CPI source and base year should be reported with any table or figure derived from these notebooks.


In [ ]:
# Shared definitions used across the analysis.
# Source-language strings appear only where they are needed to match raw data values.

from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR_CANDIDATES = [
    Path("../data/cleaned"),
    Path("data/cleaned"),
]
DATA_DIR = next(
    (candidate.resolve() for candidate in DATA_DIR_CANDIDATES if candidate.exists()),
    DATA_DIR_CANDIDATES[0].resolve(),
)
HOUSING_COL = "Housing, Water, Electricity, Gas and Other Fuels"
TEHRAN_LABELS = {"tehran", "تهران"}
OWNER_PATTERN = "own"
RENTER_PATTERN = "rent|mortg"


def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with stripped string column names."""
    out = df.copy()
    out.columns = out.columns.astype(str).str.strip()
    return out


def coerce_numeric(df: pd.DataFrame, columns: list[str], fill_missing: float | None = None) -> pd.DataFrame:
    """Coerce selected columns to numeric, optionally creating missing columns."""
    out = df.copy()
    for col in columns:
        if col not in out.columns:
            if fill_missing is None:
                raise KeyError(f"Missing required column: {col}")
            out[col] = fill_missing
        out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    """Weighted mean that ignores missing values and non-positive weights."""
    mask = values.notna() & weights.notna() & (weights > 0)
    if not mask.any():
        return np.nan
    return np.average(values[mask], weights=weights[mask])


def add_urban_rural_label(df: pd.DataFrame, source_col: str = "Urban_Rural", target_col: str = "UR") -> pd.DataFrame:
    out = df.copy()
    ur = out[source_col].astype(str).str.strip().str.lower()
    out[target_col] = pd.NA
    out.loc[ur.str.startswith("u"), target_col] = "Urban"
    out.loc[ur.str.startswith("r"), target_col] = "Rural"
    return out[out[target_col].isin(["Urban", "Rural"])].copy()


def add_tenure_group(df: pd.DataFrame, source_col: str = "Tenure", target_col: str = "tenure_group") -> pd.DataFrame:
    out = df.copy()
    tenure = out[source_col].astype(str).str.strip().str.lower()
    out[target_col] = pd.NA
    out.loc[tenure.str.contains(RENTER_PATTERN, na=False), target_col] = "Renter"
    out.loc[tenure.str.contains(OWNER_PATTERN, na=False), target_col] = "Owner"
    return out[out[target_col].isin(["Owner", "Renter"])].copy()


def add_urban_tehran_group(df: pd.DataFrame, ur_col: str = "UR", target_col: str = "UR_EXT") -> pd.DataFrame:
    out = df.copy()
    province = out["Province"].astype(str).str.strip().str.lower()
    out[target_col] = out[ur_col]
    out.loc[(out[ur_col] == "Urban") & province.isin(TEHRAN_LABELS), target_col] = "Urban_Tehran"
    return out


def infer_iran_year(file_name: str) -> int:
    stem = int(Path(file_name).stem)
    if stem >= 100:
        return 1000 + stem
    if stem >= 90:
        return 1300 + stem
    return 1400 + stem


In [ ]:
# Load the prepared analysis file.

master = clean_columns(pd.read_csv(DATA_DIR / "master.csv"))

required_columns = {
    "Year", "ID", "Urban_Rural", "Province", "Weight", "Tenure",
    "Income", "Gross_Expenditure", "housing_cost_adj",
}
missing_columns = sorted(required_columns.difference(master.columns))
if missing_columns:
    raise KeyError(f"master.csv is missing required columns: {missing_columns}")

print(f"master loaded: {master.shape[0]:,} rows x {master.shape[1]:,} columns")
print(f"years: {master['Year'].min()}-{master['Year'].max()}")


## 3. Housing Share by Geography

Housing cost is divided by gross expenditure for each household. The table reports weighted annual means for rural households, urban households, and urban Tehran.


In [15]:
# Estimate housing shares by year and geography.

import pandas as pd
import numpy as np

df = master.copy()
df.columns = df.columns.astype(str).str.strip()


for c in ["Weight", "Gross_Expenditure", "housing_cost_adj"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")


df = df[(df["Weight"] > 0) & (df["Gross_Expenditure"] > 0)].copy()


ur = df["Urban_Rural"].astype(str).str.strip().str.lower()
df["UR"] = pd.NA
df.loc[ur.str.startswith("u"), "UR"] = "Urban"
df.loc[ur.str.startswith("r"), "UR"] = "Rural"
df = df[df["UR"].isin(["Urban", "Rural"])].copy()


df["housing_share"] = (
    df["housing_cost_adj"] / df["Gross_Expenditure"]
)


prov = df["Province"].astype(str).str.strip().str.lower()

df["UR_EXT"] = df["UR"]
df.loc[
    (df["UR"] == "Urban") & (prov.isin(TEHRAN_LABELS)),
    "UR_EXT"
] = "Urban_Tehran"


tab_long = (
    df.groupby(["Year", "UR_EXT"], as_index=False)
      .apply(lambda x: pd.Series({
          "housing_share_wmean": np.average(x["housing_share"], weights=x["Weight"])
      }))
)


tab_wide = (
    tab_long.pivot(index="Year", columns="UR_EXT", values="housing_share_wmean")
            .reset_index()
            .rename_axis(None, axis=1)
            .sort_values("Year")
)

tab_wide


C:\Users\M.A\AppData\Local\Temp\ipykernel_3924\3076008069.py:50: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,Year,Rural,Urban,Urban_Tehran
0,1394,0.092965,0.130969,0.150443
1,1395,0.097856,0.131305,0.151729
2,1396,0.095922,0.126172,0.142177
3,1397,0.089611,0.124340,0.143938
4,1398,0.080008,0.112941,0.147790
5,1399,0.077852,0.110831,0.132627
6,1400,0.065659,0.103673,0.141672
7,1401,0.057601,0.105659,0.142705
8,1402,0.055695,0.109766,0.161603
9,1403,0.055129,0.115829,0.166623


The ratio output is converted to percentages for presentation.


In [7]:
# Express housing shares as percentages.

tab_percent = tab_wide.copy()


for c in ["Urban", "Rural"]:
    if c in tab_percent.columns:
        tab_percent[c] = tab_percent[c] * 100


tab_percent[["Urban", "Rural"]] = tab_percent[["Urban", "Rural"]].round(1)

tab_percent


,Year,Rural,Urban
0,1394,21.1,34.7
1,1395,22.1,35.1
2,1396,22.2,33.9
3,1397,21.5,34.4
4,1398,21.8,35.6
5,1399,22.8,36.6
6,1400,22.6,36.3
7,1401,22.9,38.1
8,1402,25.5,41.5
9,1403,27.4,43.9


## 4. Renter-to-Owner Ratio

This table compares the weighted size of the renter population with the weighted size of the owner population by year and geography.


In [7]:
# Compute renter-to-owner ratios using survey weights.

import pandas as pd
import numpy as np

df = master.copy()
df.columns = df.columns.astype(str).str.strip()


df["Weight"] = pd.to_numeric(df["Weight"], errors="coerce")
df = df[df["Weight"] > 0].copy()


ur = df["Urban_Rural"].astype(str).str.strip().str.lower()
df["UR"] = pd.NA
df.loc[ur.str.startswith("u"), "UR"] = "Urban"
df.loc[ur.str.startswith("r"), "UR"] = "Rural"
df = df[df["UR"].isin(["Urban", "Rural"])].copy()


ten = df["Tenure"].astype(str).str.strip().str.lower()
df["tenure_group"] = pd.NA
df.loc[ten.str.contains("rent|mortg"), "tenure_group"] = "Renter"
df.loc[ten.str.contains("own"), "tenure_group"] = "Owner"
df = df[df["tenure_group"].isin(["Renter", "Owner"])].copy()


prov = df["Province"].astype(str).str.strip().str.lower()
df["UR_EXT"] = df["UR"]
df.loc[
    (df["UR"] == "Urban") & (prov.isin(TEHRAN_LABELS)),
    "UR_EXT"
] = "Urban_Tehran"


w = (
    df.groupby(["Year", "UR_EXT", "tenure_group"])["Weight"]
      .sum()
      .reset_index(name="w_sum")
)


pivot = (
    w.pivot(index=["Year", "UR_EXT"], columns="tenure_group", values="w_sum")
     .reset_index()
)


pivot["renter_to_owner"] = pivot["Renter"] / pivot["Owner"]


out = (
    pivot.pivot(index="Year", columns="UR_EXT", values="renter_to_owner")
         .reset_index()
         .rename_axis(None, axis=1)
         .rename(columns={
             "Urban": "Urban_Renter_to_Owner",
             "Rural": "Rural_Renter_to_Owner",
             "Urban_Tehran": "Tehran_Renter_to_Owner"
         })
         .sort_values("Year")
)

out


,Year,Rural_Renter_to_Owner,Urban_Renter_to_Owner,Tehran_Renter_to_Owner
0,1394,0.054190,0.327455,0.517394
1,1395,0.050663,0.333015,0.593635
2,1396,0.054515,0.334762,0.567782
3,1397,0.058428,0.375947,0.547096
4,1398,0.052070,0.323956,0.504192
5,1399,0.056993,0.299990,0.474622
6,1400,0.056707,0.304768,0.478652
7,1401,0.055923,0.310949,0.505267
8,1402,0.049595,0.293664,0.519690
9,1403,0.046713,0.302146,0.472107


## 5. Housing Cost Relative to Income

For urban households, adjusted housing cost is divided by income and averaged by tenure group. This gives a direct measure of housing pressure for renters and owners.


In [16]:
# Estimate housing pressure for urban owners and renters.

import pandas as pd
import numpy as np

df = master.copy()
df.columns = df.columns.astype(str).str.strip()


for c in [
    "Weight",
    "Income",
    "housing_cost_adj"
]:
    df[c] = pd.to_numeric(df[c], errors="coerce")


ur = df["Urban_Rural"].astype(str).str.strip().str.lower()
df = df[ur.str.startswith("u")].copy()


df = df[(df["Weight"] > 0) & (df["Income"] > 0)].copy()


ten = df["Tenure"].astype(str).str.strip().str.lower()
df["tenure_group"] = pd.NA
df.loc[ten.str.contains("rent|mortg"), "tenure_group"] = "Renter"
df.loc[ten.str.contains("own"), "tenure_group"] = "Owner"
df = df[df["tenure_group"].isin(["Renter", "Owner"])].copy()


df["housing_income_ratio"] = (
    df["housing_cost_adj"] / df["Income"]
)


tab = (
    df.groupby(["Year", "tenure_group"])
      .apply(lambda x: np.average(
          x["housing_income_ratio"], weights=x["Weight"]
      ))
      .reset_index(name="ratio_wmean")
)


out = (
    tab.pivot(index="Year", columns="tenure_group", values="ratio_wmean")
       .reset_index()
       .rename_axis(None, axis=1)
       .rename(columns={
           "Renter": "Urban_Renter_%",
           "Owner": "Urban_Owner_%"
       })
       .sort_values("Year")
)

out["Urban_Renter_%"] *= 100
out["Urban_Owner_%"] *= 100


out[["Urban_Renter_%", "Urban_Owner_%"]] = out[
    ["Urban_Renter_%", "Urban_Owner_%"]
].round(1)

out


C:\Users\M.A\AppData\Local\Temp\ipykernel_3924\801782122.py:37: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: np.average(


,Year,Urban_Owner_%,Urban_Renter_%
0,1394,5.2,31.1
1,1395,5.2,31.4
2,1396,4.9,28.0
3,1397,4.5,27.7
4,1398,3.9,27.1
5,1399,3.2,27.6
6,1400,2.6,26.5
7,1401,2.2,27.5
8,1402,1.9,30.2
9,1403,1.8,30.2


## 6. Tehran Housing Share by Tenure

The same housing-share calculation is repeated for urban Tehran, separately for owners and renters.


In [20]:
# Estimate Tehran housing shares by tenure.

import pandas as pd
import numpy as np


df = master.copy()
df.columns = df.columns.astype(str).str.strip()


for c in ["Weight", "Gross_Expenditure", "housing_cost_adj"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")


df = df[(df["Weight"] > 0) & (df["Gross_Expenditure"] > 0)].copy()


ur = df["Urban_Rural"].astype(str).str.strip().str.lower()
prov = df["Province"].astype(str).str.strip().str.lower()

df = df[
    ur.str.startswith("u") &
    prov.isin(TEHRAN_LABELS)
].copy()


ten = df["Tenure"].astype(str).str.strip().str.lower()
df["tenure_group"] = np.nan
df.loc[ten.str.contains("own", na=False), "tenure_group"] = "Owner"
df.loc[ten.str.contains("rent|mortg", na=False), "tenure_group"] = "Renter"
df = df[df["tenure_group"].isin(["Owner", "Renter"])].copy()


df["housing_share"] = df["housing_cost_adj"] / df["Gross_Expenditure"]


tab = (
    df.groupby(["Year", "tenure_group"])
      .apply(lambda g: np.average(g["housing_share"], weights=g["Weight"]))
      .reset_index(name="housing_share_wmean")
)


out = (
    tab.pivot(index="Year", columns="tenure_group", values="housing_share_wmean")
       .reset_index()
       .rename_axis(None, axis=1)
       .rename(columns={
           "Owner": "Tehran_Owner_%",
           "Renter": "Tehran_Renter_%"
       })
       .sort_values("Year")
)


out[["Tehran_Owner_%", "Tehran_Renter_%"]] = (
    out[["Tehran_Owner_%", "Tehran_Renter_%"]] * 100
).round(1)

out


C:\Users\M.A\AppData\Local\Temp\ipykernel_3924\2692459446.py:37: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Owner' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[ten.str.contains("own", na=False), "tenure_group"] = "Owner"
C:\Users\M.A\AppData\Local\Temp\ipykernel_3924\2692459446.py:51: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: np.average(g["housing_share"], weights=g["Weight"]))


,Year,Tehran_Owner_%,Tehran_Renter_%
0,1394,4.2,29.1
1,1395,4.2,27.1
2,1396,3.9,24.6
3,1397,3.3,26.0
4,1398,3.1,28.6
5,1399,2.5,28.4
6,1400,2.3,28.1
7,1401,2.0,27.4
8,1402,1.8,32.5
9,1403,1.6,34.5


## 7. Rental Income Among Owners

This table measures rent received by urban owner households as a share of total household income.


In [10]:
# Measure rent income as a share of owner-household income.

import pandas as pd
import numpy as np

df = master.copy()
df.columns = df.columns.astype(str).str.strip()


for c in ["Weight", "Income", "Rent"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")


ur = df["Urban_Rural"].astype(str).str.strip().str.lower()
df = df[ur.str.startswith("u")].copy()


ten = df["Tenure"].astype(str).str.strip().str.lower()
df = df[ten.str.contains("own")].copy()


df = df[(df["Weight"] > 0) & (df["Income"] > 0)].copy()


df["rent_income_ratio"] = df["Rent"] / df["Income"]


out = (
    df.groupby("Year")
      .apply(lambda x: np.average(
          x["rent_income_ratio"], weights=x["Weight"]
      ))
      .reset_index(name="Urban_Owner_Rent_to_Income_%")
      .sort_values("Year")
)


out["Urban_Owner_Rent_to_Income_%"] *= 100
out["Urban_Owner_Rent_to_Income_%"] = out[
    "Urban_Owner_Rent_to_Income_%"
].round(2)

out


C:\Users\M.A\AppData\Local\Temp\ipykernel_23640\1234278050.py:28: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: np.average(


,Year,Urban_Owner_Rent_to_Income_%
0,1394,28.28
1,1395,27.53
2,1396,26.34
3,1397,27.55
4,1398,28.59
5,1399,28.67
6,1400,28.45
7,1401,30.35
8,1402,32.16
9,1403,33.08


## 8. Real Income and Housing Cost for Urban Renters

CPI is merged to urban renter observations so that income and adjusted housing costs can be expressed in real terms before annual weighted means are calculated.


In [ ]:
# Convert renter income and housing costs to real terms.

import pandas as pd
import numpy as np


cpi = pd.read_excel(DATA_DIR / "CPI.xlsx")
cpi.columns = cpi.columns.astype(str).str.strip()

cpi = cpi.rename(columns={

    "سال": "Year",
    "CPI": "CPI",
    "شاخص قیمت مصرف کننده (CPI)": "CPI"
})

cpi["CPI"] = pd.to_numeric(cpi["CPI"], errors="coerce")


df = master.copy()
df.columns = df.columns.astype(str).str.strip()

for c in [
    "Weight",
    "Income",
    "housing_cost_adj"
]:
    df[c] = pd.to_numeric(df[c], errors="coerce")


ur = df["Urban_Rural"].astype(str).str.strip().str.lower()
df = df[ur.str.startswith("u")].copy()


ten = df["Tenure"].astype(str).str.strip().str.lower()
df = df[ten.str.contains("rent|mortg")].copy()


df = df[
    (df["Weight"] > 0) &
    (df["Income"] > 0)
].copy()


df = df.merge(cpi[["Year", "CPI"]], on="Year", how="left")
df = df[df["CPI"].notna()].copy()


df["real_income"] = df["Income"] / df["CPI"]
df["real_housing_cost"] = (
    df["housing_cost_adj"] / df["CPI"]
)


annual = (
    df.groupby("Year")
      .apply(lambda x: pd.Series({
          "real_income_wmean": np.average(
              x["real_income"], weights=x["Weight"]
          ),
          "real_housing_cost_wmean": np.average(
              x["real_housing_cost"], weights=x["Weight"]
          )
      }))
      .reset_index()
      .sort_values("Year")
)


annual["ΔReal_Income_%"] = (
    annual["real_income_wmean"].pct_change() * 100
)

annual["ΔReal_Housing_Cost_%"] = (
    annual["real_housing_cost_wmean"].pct_change() * 100
)

out = annual[[
    "Year",
    "ΔReal_Income_%",
    "ΔReal_Housing_Cost_%"
]].round(2)

out


C:\Users\M.A\AppData\Local\Temp\ipykernel_23640\1428976860.py:64: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,Year,ΔReal_Income_%,ΔReal_Housing_Cost_%
0,1394,NaN,NaN
1,1395,4.96,0.39
2,1396,8.54,7.26
3,1397,-2.37,0.75
4,1398,-9.05,-8.76
5,1399,1.49,-0.56
6,1400,3.94,-1.52
7,1401,5.83,8.22
8,1402,6.85,16.74
9,1403,1.80,1.50


## 9. Real Annual Summary Tables

This section creates four real-value summary tables: urban renters, urban owners, Tehran renters, and Tehran owners. Income and gross expenditure are scaled to 10 million rials, while adjusted housing cost is scaled to 1 million rials.


In [31]:
# Create real weighted summary tables.

import pandas as pd
import numpy as np


cpi = pd.DataFrame([
    (1389,100.0000),(1390,126.2934),(1391,160.7169),(1392,219.5442),
    (1393,256.0029),(1394,287.9641),(1395,308.8283),(1396,333.6733),
    (1397,393.7816),(1398,550.9294),(1399,719.4815),(1400,1031.6580),
    (1401,1480.3100),(1402,2140.219429),(1403,2834.846295)
], columns=["Year","CPI"])


df = master.copy()
df.columns = df.columns.astype(str).str.strip()

for c in ["Year","Weight","Income","Gross_Expenditure","housing_cost_adj"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df[(df["Weight"] > 0) & df["Year"].notna()].copy()


ur = df["Urban_Rural"].astype(str).str.lower().str.strip()
df = df[ur.str.startswith("u")].copy()

ten = df["Tenure"].astype(str).str.lower().str.strip()
df["tenure_group"] = np.nan
df.loc[ten.str.contains("rent|mortg", na=False), "tenure_group"] = "Renter"
df.loc[ten.str.contains("own", na=False), "tenure_group"] = "Owner"
df = df[df["tenure_group"].isin(["Owner","Renter"])].copy()

prov = df["Province"].astype(str).str.lower().str.strip()
df["is_tehran"] = prov.isin(TEHRAN_LABELS)


df = df.merge(cpi, on="Year", how="left")
df = df[df["CPI"].notna()].copy()

df["Income_real"]            = df["Income"]            * (100 / df["CPI"])
df["Gross_Expenditure_real"] = df["Gross_Expenditure"] * (100 / df["CPI"])
df["housing_cost_adj_real"]  = df["housing_cost_adj"]  * (100 / df["CPI"])


def wmean(x, w):
    m = x.notna() & w.notna() & (w > 0)
    return np.average(x[m], weights=w[m]) if m.any() else np.nan

def build_table(dfg):
    out = (
        dfg.groupby("Year")
           .apply(lambda g: pd.Series({

               "Income_real_wmean":            wmean(g["Income_real"], g["Weight"]) / 10_000_000,
               "Gross_Expenditure_real_wmean": wmean(g["Gross_Expenditure_real"], g["Weight"]) / 10_000_000,
               "housing_cost_adj_real_wmean":  wmean(g["housing_cost_adj_real"], g["Weight"]) / 10_000_000,
           }))
           .reset_index()
           .sort_values("Year")
    )
    return out


df_tehran_renter = build_table(df[(df["is_tehran"]) & (df["tenure_group"]=="Renter")])
df_tehran_owner  = build_table(df[(df["is_tehran"]) & (df["tenure_group"]=="Owner")])
df_urban_renter  = build_table(df[df["tenure_group"]=="Renter"])
df_urban_owner   = build_table(df[df["tenure_group"]=="Owner"])


df_tehran_renter


C:\Users\M.A\AppData\Local\Temp\ipykernel_3924\4258281182.py:40: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Renter' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[ten.str.contains("rent|mortg", na=False), "tenure_group"] = "Renter"
C:\Users\M.A\AppData\Local\Temp\ipykernel_3924\4258281182.py:67: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({
C:\Users\M.A\AppData\Local\Temp\ipykernel_3924\4258281182.py:67: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas t

,Year,Income_real_wmean,Gross_Expenditure_real_wmean,housing_cost_adj_real_wmean
0,1394,11.410911,12.488040,3.487143
1,1395,12.183520,11.986874,2.923389
2,1396,13.337364,13.077150,3.027572
3,1397,13.672559,13.721227,3.420258
4,1398,12.628657,11.850056,3.372689
5,1399,12.043160,11.688059,3.231116
6,1400,12.824901,12.054451,3.267799
7,1401,13.394462,12.406244,3.269749
8,1402,14.412913,13.869058,4.319805
9,1403,14.761332,12.272895,3.969594


Tehran owners.


In [32]:
df_tehran_owner


,Year,Income_real_wmean,Gross_Expenditure_real_wmean,housing_cost_adj_real_wmean
0,1394,14.058541,13.822364,0.494808
1,1395,16.123365,14.960178,0.551017
2,1396,17.159806,15.349774,0.563841
3,1397,18.134114,16.150284,0.467228
4,1398,16.200108,14.561650,0.393933
5,1399,17.183237,14.423805,0.335216
6,1400,17.253675,14.465772,0.329554
7,1401,16.934739,13.857247,0.264602
8,1402,20.075333,16.971866,0.300867
9,1403,19.045472,15.264574,0.241491


Urban renters.


In [33]:
df_urban_renter


,Year,Income_real_wmean,Gross_Expenditure_real_wmean,housing_cost_adj_real_wmean
0,1394,8.678681,9.136868,2.359229
1,1395,9.106441,9.037351,2.170425
2,1396,9.885328,9.736220,2.274182
3,1397,9.651617,9.866681,2.204217
4,1398,8.775067,8.554936,2.075983
5,1399,8.901142,8.449408,2.062148
6,1400,9.256434,8.766331,2.150532
7,1401,9.800361,9.038650,2.340962
8,1402,10.471823,9.720411,2.815251
9,1403,10.654469,9.305997,2.843638


Urban owners.


In [34]:
df_urban_owner


,Year,Income_real_wmean,Gross_Expenditure_real_wmean,housing_cost_adj_real_wmean
0,1394,10.314235,9.591161,0.433315
1,1395,11.019894,9.790380,0.441174
2,1396,11.688473,10.320928,0.447943
3,1397,11.904243,10.619793,0.408502
4,1398,10.401017,9.080575,0.316617
5,1399,11.139385,9.104796,0.282639
6,1400,11.729241,9.422070,0.250346
7,1401,11.999283,9.677527,0.217535
8,1402,12.694482,9.987578,0.202461
9,1403,12.778058,9.886806,0.191490


## 10. Tenure Composition by Income Decile

Within each modified OECD income decile, the table reports the weighted share of urban households that are owners and renters.


In [21]:
# Estimate tenure composition across income deciles.

import pandas as pd
import numpy as np

df = master.copy()
df.columns = df.columns.astype(str).str.strip()


df["Weight"] = pd.to_numeric(df["Weight"], errors="coerce")
df = df[df["Weight"] > 0].copy()


ur = df["Urban_Rural"].astype(str).str.strip().str.lower()
df = df[ur.str.startswith("u")].copy()


df["Income_Decile_OECD_Modified"] = pd.to_numeric(
    df["Income_Decile_OECD_Modified"], errors="coerce"
)
df = df[df["Income_Decile_OECD_Modified"].between(1, 10)].copy()


ten = df["Tenure"].astype(str).str.strip().str.lower()
df["tenure_group"] = pd.NA
df.loc[ten.str.contains("rent|mortg"), "tenure_group"] = "Renter"
df.loc[ten.str.contains("own"), "tenure_group"] = "Owner"

df = df[df["tenure_group"].isin(["Owner", "Renter"])].copy()


w = (
    df.groupby(["Income_Decile_OECD_Modified", "tenure_group"])["Weight"]
      .sum()
      .reset_index(name="w_sum")
)


total = (
    w.groupby("Income_Decile_OECD_Modified")["w_sum"]
     .sum()
     .reset_index(name="w_total")
)

w = w.merge(total, on="Income_Decile_OECD_Modified", how="left")
w["share_pct"] = 100 * w["w_sum"] / w["w_total"]


out = (
    w.pivot(
        index="Income_Decile_OECD_Modified",
        columns="tenure_group",
        values="share_pct"
    )
    .reset_index()
    .rename_axis(None, axis=1)
    .rename(columns={
        "Income_Decile_OECD_Modified": "Decile",
        "Owner": "Owner_%",
        "Renter": "Renter_%"
    })
    .sort_values("Decile")
)


out[["Owner_%", "Renter_%"]] = out[["Owner_%", "Renter_%"]].round(1)

out


,Decile,Owner_%,Renter_%
0,1.0,66.4,33.6
1,2.0,64.0,36.0
2,3.0,64.7,35.3
3,4.0,68.4,31.6
4,5.0,70.4,29.6
5,6.0,73.5,26.5
6,7.0,77.0,23.0
7,8.0,78.1,21.9
8,9.0,80.1,19.9
9,10.0,81.5,18.5


## 11. Expenditure by Category

This table reports annual weighted mean expenditure for selected spending categories.


In [ ]:
# Compute annual weighted mean expenditure by category.

import pandas as pd
import numpy as np

df = master.copy()
df.columns = df.columns.astype(str).str.strip()


COLS = {
    "Clothing and Footwear": "Clothing and Footwear",
    "Housing, Water, Electricity, Gas and Other Fuels": "housing_cost_adj",
    "Furnishings and Household Maintenance": "Furnishings Household Equipment and Routine Household Maintenance",
    "Health": "Health",
    "Transport": "Transport",
    "Information and Communication": "Information and Communication",
    "Recreation and Culture": "Recreation and Culture",
    "Education Services": "Education Services",
    "Food and Non-Alcoholic Beverages": "Food and Non-Alcoholic Beverages",
}


df["Weight"] = pd.to_numeric(df["Weight"], errors="coerce")
for fa, en in COLS.items():
    df[en] = pd.to_numeric(df[en], errors="coerce")


df = df[df["Weight"] > 0].copy()

def wmean(s, w):
    m = s.notna() & w.notna() & (w > 0)
    if m.sum() == 0:
        return np.nan
    return np.average(s[m], weights=w[m])


out = df.groupby("Year").apply(
    lambda g: pd.Series({fa: wmean(g[en], g["Weight"]) for fa, en in COLS.items()})
).reset_index().sort_values("Year")

out


C:\Users\M.A\AppData\Local\Temp\ipykernel_23640\2154888463.py:37: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  out = df.groupby("Year").apply(


,Year,پوشاک و کفش,مسکن، آب، فاضلاب، سوخت و روشنایی,مبلمان، لوازم خانگی، و نگهداری معمول آن‌ها,بهداشت و درمان,حمل و نقل,اطلاعات و ارتباطات,خدمات فرهنگی و تفریحات,آموزش و تحصیل,مواد خوراکی و نوشیدنی‌های غیر الکلی
0,1394,1.027692e+07,7.207240e+07,8.560916e+06,1.681038e+07,2.161857e+07,6.062729e+06,3.754451e+06,3.797743e+06,5.794552e+07
1,1395,1.062855e+07,7.945863e+07,8.939016e+06,1.784925e+07,2.386257e+07,6.415577e+06,3.739965e+06,4.043073e+06,6.116759e+07
2,1396,1.230531e+07,8.994096e+07,1.102001e+07,2.125233e+07,2.960159e+07,7.331954e+06,4.727944e+06,4.672998e+06,6.966518e+07
3,1397,1.312265e+07,1.099353e+08,1.293196e+07,2.707834e+07,3.499179e+07,8.136286e+06,5.620093e+06,5.016007e+06,8.481885e+07
4,1398,1.410529e+07,1.425409e+08,1.526509e+07,3.010390e+07,3.643821e+07,9.475605e+06,5.235047e+06,5.156833e+06,1.070512e+08
5,1399,1.959973e+07,1.911024e+08,2.152238e+07,3.561965e+07,4.541739e+07,1.368368e+07,4.170505e+06,5.303764e+06,1.485725e+08
6,1400,3.464476e+07,2.780883e+08,3.233449e+07,5.579829e+07,6.604759e+07,1.835974e+07,6.538308e+06,6.888279e+06,2.257369e+08
7,1401,4.783488e+07,4.378691e+08,4.189022e+07,7.343770e+07,9.524133e+07,2.018021e+07,1.092669e+07,1.008329e+07,3.482020e+08
8,1402,6.385760e+07,7.337816e+08,5.540872e+07,1.063928e+08,1.458816e+08,2.299674e+07,1.876853e+07,1.382967e+07,4.684285e+08
9,1403,8.490385e+07,9.937751e+08,6.469091e+07,1.344669e+08,1.760517e+08,2.827657e+07,2.853040e+07,2.137285e+07,5.782720e+08


## 12. Income Deciles Before and After Housing Costs

This section compares household rank in the income distribution before and after subtracting housing costs. Deciles are assigned within each year using survey weights.


In [ ]:
# Compare income deciles before and after housing costs.

import pandas as pd
import numpy as np

df = master.copy()
df.columns = df.columns.astype(str).str.strip()


for c in ["Year", "Weight", "Income", "housing_cost_adj"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df[(df["Weight"] > 0) & df["Year"].notna()].copy()


df["housing_cost"] = pd.to_numeric(df["housing_cost_adj"], errors="coerce")
df["income_after_housing"] = df["Income"] - df["housing_cost"]


df.loc[df["income_after_housing"] < 0, "income_after_housing"] = 0


def assign_weighted_deciles(values: pd.Series, weights: pd.Series) -> pd.Series:
    v = values.to_numpy()
    w = weights.to_numpy()

    ok = np.isfinite(v) & np.isfinite(w) & (w > 0)
    out = np.full(len(v), np.nan)

    if ok.sum() == 0:
        return pd.Series(out, index=values.index)

    idx = np.where(ok)[0]
    order = idx[np.argsort(v[ok], kind="mergesort")]

    w_sorted = w[order]
    cum = np.cumsum(w_sorted) / w_sorted.sum()


    d_sorted = np.ceil(10 * cum).astype(int)
    d_sorted[d_sorted < 1] = 1
    d_sorted[d_sorted > 10] = 10

    out[order] = d_sorted
    return pd.Series(out, index=values.index)


df["decile_before"] = df.groupby("Year", group_keys=False).apply(
    lambda g: assign_weighted_deciles(g["Income"].fillna(0), g["Weight"])
)
df["decile_after"] = df.groupby("Year", group_keys=False).apply(
    lambda g: assign_weighted_deciles(g["income_after_housing"].fillna(0), g["Weight"])
)


ur = df["Urban_Rural"].astype(str).str.strip().str.lower()
dfu = df[ur.str.startswith("u")].copy()

ten = dfu["Tenure"].astype(str).str.strip().str.lower()
dfu["tenure_group"] = pd.NA
dfu.loc[ten.str.contains("rent|mortg"), "tenure_group"] = "Renter"
dfu.loc[ten.str.contains("own"), "tenure_group"] = "Owner"
dfu = dfu[dfu["tenure_group"].isin(["Owner", "Renter"])].copy()


dfu["decile_drop_ratio"] = (
    (dfu["decile_before"] - dfu["decile_after"]).clip(lower=0) / dfu["decile_before"]
)


tab = (
    dfu.dropna(subset=["decile_drop_ratio", "decile_before", "decile_after"])
       .groupby(["Year", "tenure_group"])
       .apply(lambda g: np.average(g["decile_drop_ratio"], weights=g["Weight"]) * 100)
       .reset_index(name="Decile_Drop_%")
)


out = (
    tab.pivot(index="Year", columns="tenure_group", values="Decile_Drop_%")
       .reset_index()
       .rename_axis(None, axis=1)
       .rename(columns={"Owner": "Owner_DecileDrop_%", "Renter": "Renter_DecileDrop_%"})
       .sort_values("Year")
)

out[["Owner_DecileDrop_%", "Renter_DecileDrop_%"]] = out[["Owner_DecileDrop_%", "Renter_DecileDrop_%"]].round(1)

out


C:\Users\M.A\AppData\Local\Temp\ipykernel_23640\675564440.py:46: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df["decile_before"] = df.groupby("Year", group_keys=False).apply(
C:\Users\M.A\AppData\Local\Temp\ipykernel_23640\675564440.py:49: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df["decile_after"] = df.groupby("Year", group_keys=False).apply(
C:\Users\M.A\AppData\Local\Temp\ipykernel_23640\675564440.py:73: Fu

,Year,Owner_DecileDrop_%,Renter_DecileDrop_%
0,1394,6.3,14.7
1,1395,6.0,13.3
2,1396,5.4,12.7
3,1397,5.7,12.8
4,1398,6.1,12.9
5,1399,6.3,13.1
6,1400,6.4,12.0
7,1401,7.2,12.5
8,1402,7.4,14.0
9,1403,7.8,13.6


## 13. Housing Poverty Thresholds

Urban renters are classified as housing poor when adjusted housing costs exceed 30 percent or 40 percent of income. The output reports weighted annual rates at both thresholds.


In [ ]:
# Estimate housing-poverty rates for urban renters.

import pandas as pd
import numpy as np

df = master.copy()
df.columns = df.columns.astype(str).str.strip()


for c in ["Weight", "Income", "housing_cost_adj"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")


ur = df["Urban_Rural"].astype(str).str.strip().str.lower()
df = df[ur.str.startswith("u")].copy()


ten = df["Tenure"].astype(str).str.strip().str.lower()
df = df[ten.str.contains("rent|mortg")].copy()


df = df[(df["Weight"] > 0) & (df["Income"] > 0)].copy()


df["housing_pressure"] = df["housing_cost_adj"] / df["Income"]


df["poor_30"] = (df["housing_pressure"] >= 0.30).astype(int)
df["poor_40"] = (df["housing_pressure"] >= 0.40).astype(int)


out = (
    df.groupby("Year")
      .apply(lambda g: pd.Series({
          "Housing_Poverty_30_%": 100 * np.average(g["poor_30"], weights=g["Weight"]),
          "Housing_Poverty_40_%": 100 * np.average(g["poor_40"], weights=g["Weight"]),
      }))
      .reset_index()
      .sort_values("Year")
)

out[["Housing_Poverty_30_%", "Housing_Poverty_40_%"]] = out[
    ["Housing_Poverty_30_%", "Housing_Poverty_40_%"]
].round(1)

out


C:\Users\M.A\AppData\Local\Temp\ipykernel_23640\70308554.py:32: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


,Year,Housing_Poverty_30_%,Housing_Poverty_40_%
0,1394,67.3,43.3
1,1395,65.1,41.4
2,1396,62.7,37.8
3,1397,62.0,38.9
4,1398,62.2,41.6
5,1399,59.8,38.6
6,1400,55.8,34.2
7,1401,60.4,38.4
8,1402,64.9,41.8
9,1403,66.4,44.2


## 14. Non-Cash Employment Income

The yearly files are read again to extract `NonCash_Employment_Income`, which is then used in the following section.


In [21]:
# Extract non-cash employment income from yearly files.

import pandas as pd
from pathlib import Path


FILES = [
    "94.csv","95.csv","96.csv","97.csv","98.csv","99.csv",
    "400.csv","401.csv","402.csv","403.csv",
]
FILES = [f for f in FILES if Path(f).exists()]

if not FILES:
    raise FileNotFoundError("No yearly CSV files were found next to the notebook.")


TARGET = "NonCash_Employment_Income"
ALIASES = [
    "NonCash_Employment_Income",
    "NonCash Employment Income",
    "NonCash_Employment Income",
    "Noncash_Employment_Income",
    "NonCash_EmploymentIncome",
]


KEYS = ["Year", "ID", "Weight", "Urban_Rural", "Tenure"]


rows = []
missing_files = []

for fname in FILES:
    df = pd.read_csv(DATA_DIR / fname)
    df.columns = df.columns.astype(str).str.strip()


    if "Year" not in df.columns:
        yy = int(Path(fname).stem)
        if yy >= 100:
            year = 1000 + yy
        elif yy >= 90:
            year = 1300 + yy
        else:
            year = 1400 + yy
        df["Year"] = year


    col = next((c for c in ALIASES if c in df.columns), None)
    if col is None:
        missing_files.append(fname)
        continue


    keep = [c for c in KEYS if c in df.columns] + [col]
    tmp = df[keep].copy()


    if col != TARGET:
        tmp = tmp.rename(columns={col: TARGET})

    tmp["__source_file__"] = fname
    rows.append(tmp)

if not rows:
    raise ValueError("NonCash_Employment_Income was not found in any yearly file.")

noncash_emp = pd.concat(rows, ignore_index=True)

print("✅ extracted dataframe: noncash_emp")
print("rows:", len(noncash_emp), "| cols:", noncash_emp.shape[1])

if missing_files:
    print("\nFiles without the target column:")
    for f in missing_files:
        print(" -", f)

noncash_emp.head()


C:\Users\M.A\AppData\Local\Temp\ipykernel_23640\3851349803.py:39: DtypeWarning: Columns (145) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(fname)
C:\Users\M.A\AppData\Local\Temp\ipykernel_23640\3851349803.py:39: DtypeWarning: Columns (145) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(fname)
C:\Users\M.A\AppData\Local\Temp\ipykernel_23640\3851349803.py:39: DtypeWarning: Columns (139) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(fname)
C:\Users\M.A\AppData\Local\Temp\ipykernel_23640\3851349803.py:39: DtypeWarning: Columns (139) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(fname)


✅ extracted dataframe: noncash_emp
rows: 380532 | cols: 7


,Year,ID,Weight,Urban_Rural,Tenure,NonCash_Employment_Income,__source_file__
0,1394,10001000116,1216.8722,Urban,Owned_Estate_Land,0.0,94.csv
1,1394,10001000117,1216.8722,Urban,Owned_Estate_Land,0.0,94.csv
2,1394,10001000120,1216.8722,Urban,Owned_Estate_Land,0.0,94.csv
3,1394,10001000123,1216.8722,Urban,Owned_Estate_Land,0.0,94.csv
4,1394,10001000126,1216.8722,Urban,Owned_Estate_Land,0.0,94.csv


## 15. Non-Cash Employment Income Among Housing-Poor Renters

This table focuses on urban renters and summarizes the employment-income proxy among households above the housing-poverty thresholds.


In [ ]:
# Summarize non-cash employment income among housing-poor renters.

import numpy as np
import pandas as pd


df = noncash_emp.copy()
df.columns = df.columns.astype(str).str.strip()


for c in ["Weight", "NonCash_Employment_Income"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")


ur = df["Urban_Rural"].astype(str).str.strip().str.lower()
df = df[ur.str.startswith("u")].copy()

ten = df["Tenure"].astype(str).str.strip().str.lower()
df = df[ten.str.contains("rent|mortg")].copy()


df = df[df["Weight"] > 0].copy()


base = master[[
    "Year", "ID",
    "Income",
    "housing_cost_adj"
]].copy()

base.columns = base.columns.astype(str).str.strip()

for c in ["Income", "housing_cost_adj"]:
    base[c] = pd.to_numeric(base[c], errors="coerce")


df = df.merge(
    base,
    on=["Year", "ID"],
    how="left"
)

df = df[(df["Income"] > 0)].copy()

df["housing_pressure"] = (
    df["housing_cost_adj"] / df["Income"]
)


df["poor_30"] = df["housing_pressure"] >= 0.30
df["poor_40"] = df["housing_pressure"] >= 0.40


df["employed"] = df["NonCash_Employment_Income"] > 0


out = (
    df.groupby("Year")
      .apply(lambda g: pd.Series({
          "Employed_Among_Poor_30_%": (
              100 * np.average(
                  g.loc[g["poor_30"], "employed"],
                  weights=g.loc[g["poor_30"], "Weight"]
              ) if g["poor_30"].any() else np.nan
          ),
          "Employed_Among_Poor_40_%": (
              100 * np.average(
                  g.loc[g["poor_40"], "employed"],
                  weights=g.loc[g["poor_40"], "Weight"]
              ) if g["poor_40"].any() else np.nan
          ),
      }))
      .reset_index()
      .sort_values("Year")
)

out[["Employed_Among_Poor_30_%", "Employed_Among_Poor_40_%"]] = out[
    ["Employed_Among_Poor_30_%", "Employed_Among_Poor_40_%"]
].round(1)

out


C:\Users\M.A\AppData\Local\Temp\ipykernel_23640\2993830465.py:70: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


,Year,Employed_Among_Poor_30_%,Employed_Among_Poor_40_%
0,1394,28.6,24.8
1,1395,26.3,21.0
2,1396,24.4,18.6
3,1397,22.4,16.9
4,1398,22.4,19.4
5,1399,22.4,18.5
6,1400,22.0,16.6
7,1401,26.4,21.4
8,1402,23.3,18.6
9,1403,25.0,18.1


## 16. Disposable Income After Housing

Disposable income after housing is defined as income minus adjusted housing cost. Negative values are set to zero before weighted annual averages are estimated by tenure.


In [42]:
# Estimate disposable income after housing by tenure.

import pandas as pd
import numpy as np

df = master.copy()
df.columns = df.columns.astype(str).str.strip()


for c in [
    "Weight",
    "Income",
    "housing_cost_adj"
]:
    df[c] = pd.to_numeric(df[c], errors="coerce")


ur = df["Urban_Rural"].astype(str).str.strip().str.lower()
df = df[ur.str.startswith("u")].copy()


ten = df["Tenure"].astype(str).str.strip().str.lower()
df["tenure_group"] = pd.NA
df.loc[ten.str.contains("own"), "tenure_group"] = "Owner"
df.loc[ten.str.contains("rent|mortg"), "tenure_group"] = "Renter"
df = df[df["tenure_group"].isin(["Owner", "Renter"])].copy()


df["disposable_income"] = (
    df["Income"] - df["housing_cost_adj"]
)


df.loc[df["disposable_income"] < 0, "disposable_income"] = 0


df = df[
    (df["Weight"] > 0) &
    (df["Income"] > 0)
].copy()


tab = (
    df.groupby(["Year", "tenure_group"])
      .apply(lambda g: np.average(
          g["disposable_income"], weights=g["Weight"]
      ))
      .reset_index(name="Disposable_Income_WMean")
)


out = (
    tab.pivot(index="Year", columns="tenure_group", values="Disposable_Income_WMean")
       .reset_index()
       .rename_axis(None, axis=1)
       .rename(columns={
           "Owner": "Owner_Disposable_Income",
           "Renter": "Renter_Disposable_Income"
       })
       .sort_values("Year")
)


for c in ["Owner_Disposable_Income", "Renter_Disposable_Income"]:
    out[c] = out[c] / 10_000_000


out[["Owner_Disposable_Income", "Renter_Disposable_Income"]] = (
    out[["Owner_Disposable_Income", "Renter_Disposable_Income"]]
    .round(1)
)

out


C:\Users\M.A\AppData\Local\Temp\ipykernel_3924\1026794863.py:55: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: np.average(


,Year,Owner_Disposable_Income,Renter_Disposable_Income
0,1394,28.5,18.2
1,1395,32.7,21.5
2,1396,37.5,25.4
3,1397,45.3,29.4
4,1398,55.6,37.0
5,1399,78.1,49.4
6,1400,118.4,73.5
7,1401,174.4,110.5
8,1402,267.4,164.1
9,1403,356.8,221.9


## 17. Ownership and Renting at the Bottom and Top of the Distribution

This section compares owner and renter shares in selected income-decile bands, especially the first and tenth deciles and broader bottom/top groups.


In [40]:
# Compare tenure shares in lower and upper decile groups.

import pandas as pd
import numpy as np

df = master.copy()
df.columns = df.columns.astype(str).str.strip()


df["Weight"] = pd.to_numeric(df["Weight"], errors="coerce")
df["Income_Decile_OECD_Modified"] = pd.to_numeric(df["Income_Decile_OECD_Modified"], errors="coerce")


df = df[(df["Weight"] > 0)].copy()

ur = df["Urban_Rural"].astype(str).str.strip().str.lower()
df = df[ur.str.startswith("u")].copy()

df = df[df["Income_Decile_OECD_Modified"].between(1, 10)].copy()


ten = df["Tenure"].astype(str).str.strip().str.lower()
df["tenure_group"] = pd.NA
df.loc[ten.str.contains("rent|mortg", na=False), "tenure_group"] = "Renter"
df.loc[ten.str.contains("own", na=False), "tenure_group"] = "Owner"
df = df[df["tenure_group"].isin(["Owner", "Renter"])].copy()


def shares_in_group(g: pd.DataFrame) -> pd.Series:
    w_owner = g.loc[g["tenure_group"] == "Owner", "Weight"].sum()
    w_renter = g.loc[g["tenure_group"] == "Renter", "Weight"].sum()
    w_total = w_owner + w_renter
    if w_total == 0:
        return pd.Series({"Owner_%": np.nan, "Renter_%": np.nan})
    return pd.Series({
        "Owner_%": 100 * w_owner / w_total,
        "Renter_%": 100 * w_renter / w_total
    })


df_tb = df.copy()
df_tb["decile_band"] = pd.NA
df_tb.loc[df_tb["Income_Decile_OECD_Modified"].between(1, 3), "decile_band"] = "Bottom_1_3"
df_tb.loc[df_tb["Income_Decile_OECD_Modified"].between(8, 10), "decile_band"] = "Top_8_10"
df_tb = df_tb[df_tb["decile_band"].isin(["Bottom_1_3", "Top_8_10"])].copy()

tab_tb = (
    df_tb.groupby(["Year", "decile_band"])
         .apply(shares_in_group)
         .reset_index()
)

df_share_topbottom = (
    tab_tb.pivot(index="Year", columns="decile_band", values=["Owner_%", "Renter_%"])
          .reset_index()
)


df_share_topbottom.columns = [
    "Year" if c[0] == "Year" else f"{c[1]}_{c[0]}"
    for c in df_share_topbottom.columns
]


df_share_topbottom = df_share_topbottom[[
    "Year",
    "Bottom_1_3_Owner_%", "Bottom_1_3_Renter_%",
    "Top_8_10_Owner_%", "Top_8_10_Renter_%"
]].sort_values("Year")

for c in ["Bottom_1_3_Owner_%", "Bottom_1_3_Renter_%", "Top_8_10_Owner_%", "Top_8_10_Renter_%"]:
    df_share_topbottom[c] = df_share_topbottom[c].round(1)


df_110 = df[df["Income_Decile_OECD_Modified"].isin([1, 10])].copy()
df_110["decile_band"] = df_110["Income_Decile_OECD_Modified"].map({1: "Decile_1", 10: "Decile_10"})

tab_110 = (
    df_110.groupby(["Year", "decile_band"])
          .apply(shares_in_group)
          .reset_index()
)

df_share_decile1_10 = (
    tab_110.pivot(index="Year", columns="decile_band", values=["Owner_%", "Renter_%"])
           .reset_index()
)

df_share_decile1_10.columns = [
    "Year" if c[0] == "Year" else f"{c[1]}_{c[0]}"
    for c in df_share_decile1_10.columns
]

df_share_decile1_10 = df_share_decile1_10[[
    "Year",
    "Decile_1_Owner_%", "Decile_1_Renter_%",
    "Decile_10_Owner_%", "Decile_10_Renter_%"
]].sort_values("Year")

for c in ["Decile_1_Owner_%", "Decile_1_Renter_%", "Decile_10_Owner_%", "Decile_10_Renter_%"]:
    df_share_decile1_10[c] = df_share_decile1_10[c].round(1)


df_share_topbottom,


C:\Users\M.A\AppData\Local\Temp\ipykernel_3924\4140936593.py:57: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(shares_in_group)
C:\Users\M.A\AppData\Local\Temp\ipykernel_3924\4140936593.py:90: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(shares_in_group)


(   Year  Bottom_1_3_Owner_%  Bottom_1_3_Renter_%  Top_8_10_Owner_%  \
 0  1394                66.5                 33.5              79.1   
 1  1395                66.0                 34.0              77.5   
 2  1396                64.6                 35.4              78.4   
 3  1397                62.6                 37.4              77.5   
 4  1398                65.5                 34.5              79.0   
 5  1399                66.4                 33.6              81.5   
 6  1400                65.4                 34.6              81.7   
 7  1401                64.1                 35.9              81.2   
 8  1402                63.8                 36.2              81.4   
 9  1403                64.0                 36.0              81.5   
 
    Top_8_10_Renter_%  
 0               20.9  
 1               22.5  
 2               21.6  
 3               22.5  
 4               21.0  
 5               18.5  
 6               18.3  
 7               18.8  
 

Owner and renter shares for income deciles 1 and 10.


In [41]:
df_share_decile1_10


,Year,Decile_1_Owner_%,Decile_1_Renter_%,Decile_10_Owner_%,Decile_10_Renter_%
0,1394,69.9,30.1,78.5,21.5
1,1395,63.7,36.3,80.1,19.9
2,1396,65.2,34.8,78.1,21.9
3,1397,63.4,36.6,80.6,19.4
4,1398,66.1,33.9,81.3,18.7
5,1399,69.9,30.1,83.6,16.4
6,1400,67.8,32.2,84.2,15.8
7,1401,64.9,35.1,82.5,17.5
8,1402,66.2,33.8,82.8,17.2
9,1403,67.0,33.0,82.5,17.5


## 18. Expenditure-Inflation Exposure

Household budget shares for food, housing, transport, and health are combined with annual category-specific inflation rates. The resulting measure approximates exposure to inflation through each household's expenditure structure, with separate outputs for Tehran and other urban areas.


In [46]:
# Combine budget shares with category inflation rates.

import pandas as pd
import numpy as np


inflation = pd.DataFrame({
    "Year": [1394,1395,1396,1397,1398,1399,1400,1401,1402,1403],
    "Food_inf":      [9.7, 7.6,12.3,38.4,42.7,38.7,52.1,70.2,41.4,28.0],
    "Housing_inf":   [14.5,6.0,7.2,17.4,23.9,25.8,26.8,32.4,39.1,39.7],
    "Transport_inf": [10.0,5.3,4.9,28.1,47.2,66.8,37.7,35.7,42.1,27.8],
    "Health_inf":    [15.0,9.1,7.2,17.3,25.9,29.7,38.6,43.5,43.3,29.2],
})


df = master.copy()
df.columns = df.columns.astype(str).str.strip()


for c in [
    "Year","Weight","Gross_Expenditure",
    "Food and Non-Alcoholic Beverages",
    "housing_cost_adj",
    "Transport","Health"
]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df[
    (df["Weight"] > 0) &
    (df["Gross_Expenditure"] > 0) &
    (df["Year"].between(1394,1403))
].copy()


ur = df["Urban_Rural"].astype(str).str.strip().str.lower()
df = df[ur.str.startswith("u")].copy()

prov = df["Province"].astype(str).str.strip().str.lower()
df["city_group"] = np.where(
    prov.isin(TEHRAN_LABELS),
    "Tehran",
    "Other_Cities"
)


ten = df["Tenure"].astype(str).str.strip().str.lower()
df["tenure_group"] = pd.NA
df.loc[ten.str.contains("rent|mortg", na=False), "tenure_group"] = "Renter"
df.loc[ten.str.contains("own", na=False), "tenure_group"] = "Owner"
df = df[df["tenure_group"].isin(["Owner","Renter"])].copy()


df = df.merge(inflation, on="Year", how="left")
df = df[df["Food_inf"].notna()].copy()


df["w_food"]      = df["Food and Non-Alcoholic Beverages"] / df["Gross_Expenditure"]
df["w_housing"]   = df["housing_cost_adj"] / df["Gross_Expenditure"]
df["w_transport"] = df["Transport"] / df["Gross_Expenditure"]
df["w_health"]    = df["Health"] / df["Gross_Expenditure"]

for c in ["w_food","w_housing","w_transport","w_health"]:
    df[c] = df[c].fillna(0)


df["experienced_inflation"] = (
    df["w_food"]      * df["Food_inf"] +
    df["w_housing"]   * df["Housing_inf"] +
    df["w_transport"] * df["Transport_inf"] +
    df["w_health"]    * df["Health_inf"]
)


tab = (
    df.groupby(["Year","city_group","tenure_group"])
      .apply(lambda g: np.average(
          g["experienced_inflation"], weights=g["Weight"]
      ))
      .reset_index(name="Experienced_Inflation_%")
      .round(2)
)


df_expinf_tehran = (
    tab[tab["city_group"]=="Tehran"]
    .pivot(index="Year", columns="tenure_group", values="Experienced_Inflation_%")
    .reset_index()
    .rename_axis(None, axis=1)
    .rename(columns={
        "Owner":"Owner_Inflation_%",
        "Renter":"Renter_Inflation_%"
    })
    .sort_values("Year")
)


df_expinf_other_cities = (
    tab[tab["city_group"]=="Other_Cities"]
    .pivot(index="Year", columns="tenure_group", values="Experienced_Inflation_%")
    .reset_index()
    .rename_axis(None, axis=1)
    .rename(columns={
        "Owner":"Owner_Inflation_%",
        "Renter":"Renter_Inflation_%"
    })
    .sort_values("Year")
)


df_expinf_tehran


C:\Users\M.A\AppData\Local\Temp\ipykernel_3924\500123713.py:90: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: np.average(


,Year,Owner_Inflation_%,Renter_Inflation_%
0,1394,4.30,7.71
1,1395,2.75,4.04
2,1396,3.54,5.10
3,1397,11.37,15.75
4,1398,13.29,19.30
5,1399,13.47,20.85
6,1400,15.27,23.70
7,1401,19.48,30.11
8,1402,12.45,25.30
9,1403,7.52,20.93


Urban households outside Tehran.


In [45]:
df_expinf_other_cities


,Year,Owner_Inflation_%,Renter_Inflation_%
0,1394,5.15,7.99
1,1395,3.35,4.45
2,1396,4.56,5.74
3,1397,14.73,17.29
4,1398,18.34,22.03
5,1399,18.65,23.30
6,1400,21.42,26.32
7,1401,27.41,34.01
8,1402,17.74,28.09
9,1403,11.76,23.19
